# STEP 1 Evidence, Logic & Capacity Audit — V2

This notebook is a **constraint and provenance system**, not an automated scientific reviewer.

The original version had two conceptual weaknesses:

1. It relied too heavily on prose/regex detection.
2. It used one proposition status to represent several distinct scientific questions.

V2 separates:

- **literature support** — what the external evidence establishes;
- **construct validity** — whether the operationalisation measures the claimed construct;
- **design identification** — what the proposed/randomised comparison can identify;
- **novelty status** — whether the contribution remains genuinely unresolved;
- **implementation status** — whether source code/materials/task details are known;
- **feasibility status** — whether the programme is executable with actual resources.

This prevents semantic patches. A construct can be renamed without changing its operationalisation, but that will not change its construct-validity or design-identification status.

## Core logical flow

**Search coverage → verified source → atomic evidence claim → atomic proposition → construct/benchmark → estimand/identification → contradiction/boundary conditions → novelty → architecture impact → closure gate**

Architecture does not determine which literature is searched.

## 1. Install

In [ ]:
!pip -q install pandas numpy requests rapidfuzz openpyxl biopython networkx scikit-learn

## 2. Imports, vocabularies, and schemas

In [ ]:
import pandas as pd
import numpy as np
import re, json, os, math, requests
from rapidfuzz import fuzz
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

LITERATURE_STATUS = {"SUPPORTED","PARTLY SUPPORTED","UNSUPPORTED","CONTRADICTED","UNRESOLVED"}
VALIDITY_STATUS = {"ADEQUATE","PARTIAL","INADEQUATE","UNRESOLVED","NOT_APPLICABLE"}
IDENTIFICATION_STATUS = {"IDENTIFIED","PARTIALLY_IDENTIFIED","NOT_IDENTIFIED","UNRESOLVED","NOT_APPLICABLE"}
NOVELTY_STATUS = {"ESTABLISHED","ADJACENT_ONLY","PARTLY_OPEN","DIRECTLY_OPEN","UNRESOLVED","NOT_APPLICABLE"}
FEASIBILITY_STATUS = {"FEASIBLE","CONDITIONAL","NOT_FEASIBLE","UNRESOLVED","NOT_APPLICABLE"}

CLAIM_TYPES = {
    "descriptive",
    "association",
    "total_policy_causal",
    "component_causal",
    "mediation",
    "moderation",
    "prediction",
    "normative",
    "measurement_validity",
    "reliability",
    "persistence",
    "recovery",
    "transfer",
    "individual_difference",
    "neural_representation",
    "neural_causal",
    "novelty",
    "algorithm_mechanics",
    "feasibility"
}

DIRECTNESS = {"direct","close analogue","adjacent","highly indirect"}
SUPPORT_RELATION = {"supports","counters","qualifies","background"}
EVIDENCE_TYPES = {
    "randomized human experiment","non-randomized human experiment",
    "observational human study","longitudinal field experiment",
    "laboratory experiment","computational simulation",
    "recommender-system simulation","methodological/statistical paper",
    "psychometric paper","cognitive model","neural study","review",
    "systematic review","meta-analysis","theoretical paper",
    "conference experiment","preregistration","source code/material"
}

SEARCH_COLUMNS = [
    "search_id","domain","database","date_searched","query","filters",
    "direct_or_adjacent","results_screened","retained_ids",
    "backward_checked","forward_checked","coverage_status","notes"
]

SOURCE_COLUMNS = [
    "evidence_id","citation","title","authors","year","doi","authoritative_url",
    "peer_reviewed","primary_source_verified","full_text_checked",
    "evidence_type","preregistered","sample_n","design",
    "manipulation","comparator","outcome","search_domain","quality_flags","notes"
]

EVIDENCE_CLAIM_COLUMNS = [
    "evidence_claim_id","evidence_id","proposition_id","support_relation",
    "exact_result_or_method","what_it_establishes","what_it_does_not_establish",
    "directness","population_match","task_match","manipulation_match","outcome_match",
    "boundary_condition","extractor_notes"
]

PROPOSITION_COLUMNS = [
    "proposition_id","canonical_claim","claim_type","construct_id",
    "operationalisation","estimand","time_horizon","transfer_dimensions",
    "normative_benchmark_id","required_interaction","validation_scheme",
    "literature_status","construct_validity_status","design_identification_status",
    "novelty_status","confidence","load_bearing","affected_studies",
    "strongest_justified_claim","stronger_claim_not_justified","notes"
]

CONSTRUCT_COLUMNS = [
    "construct_id","construct_name","literature_definition",
    "proposed_operationalisation","normative_benchmark",
    "benchmark_source_ids","validity_source_ids","reliability_source_ids",
    "compositional_or_constrained","known_failure_modes","validity_status","notes"
]

STUDY_COLUMNS = [
    "study_id","study_name","scientific_question","primary_estimand",
    "randomized_element","primary_outcome","claimed_inference",
    "dependency_proposition_ids","scientific_priority",
    "independent_workpackage_justification","identification_status",
    "feasibility_status","notes"
]

DEPENDENCY_COLUMNS = [
    "from_id","to_id","relation",
    "relation_type","load_bearing","notes"
]

CONTRADICTION_COLUMNS = [
    "contradiction_id","proposition_a","proposition_b",
    "apparent_conflict","boundary_condition","resolved_or_unresolved",
    "evidence_ids_a","evidence_ids_b","notes"
]

RECALIBRATION_COLUMNS = [
    "decision_id","previous_conclusion","current_judgement",
    "reason","evidence_or_logic_ids","affected_studies"
]

## 3. Create blank templates

In [ ]:
def make_templates(prefix="/content"):
    schemas = {
        "searches": SEARCH_COLUMNS,
        "sources": SOURCE_COLUMNS,
        "evidence_claims": EVIDENCE_CLAIM_COLUMNS,
        "propositions": PROPOSITION_COLUMNS,
        "constructs": CONSTRUCT_COLUMNS,
        "studies": STUDY_COLUMNS,
        "dependencies": DEPENDENCY_COLUMNS,
        "contradictions": CONTRADICTION_COLUMNS,
        "recalibration": RECALIBRATION_COLUMNS,
    }
    paths = {}
    for name, cols in schemas.items():
        path = f"{prefix}/{name}_template.csv"
        pd.DataFrame(columns=cols).to_csv(path, index=False)
        paths[name] = path
    return paths

template_paths = make_templates()
template_paths

## 4. Capacity map: what Python can and cannot do

### Python can reliably help with

- schema completeness;
- source-ID provenance;
- duplicate detection;
- DOI/URL presence checks;
- claim-to-evidence traceability;
- explicit claim-type requirements;
- dependency graphs and circular dependencies;
- bounded novelty-language checks;
- semantic-relabel candidates;
- contradictory-status candidates;
- search-domain coverage;
- missing boundary conditions;
- exportable audit trails;
- later simulation/model-recovery work.

### Python cannot determine

- whether a construct is scientifically valid;
- whether a paper's methods really instantiate the same mechanism;
- whether a yoked control truly solves the causal problem;
- whether a result is normatively maladaptive;
- whether a neural effect is theoretically informative;
- whether a literature search is genuinely exhaustive;
- whether a full PhD programme is feasible at the eventual host institution.

### Workaround

Python should enforce **what must be made explicit**, then a source-grounded human/LLM review adjudicates the substance. The code must never automatically convert a flag into a scientific conclusion.

## 5. Utility functions

In [ ]:
def split_ids(x):
    if pd.isna(x) or not str(x).strip():
        return []
    return [v.strip() for v in re.split(r"[;,]", str(x)) if v.strip()]

def truthy(x):
    return str(x).strip().lower() in {"1","true","yes","y"}

def empty(x):
    return pd.isna(x) or not str(x).strip()

def norm_text(s):
    return re.sub(r"\s+"," ",re.sub(r"[^a-z0-9 ]"," ",str(s).lower())).strip()

def valid_enum(df, col, allowed):
    out = []
    if col not in df.columns:
        return pd.DataFrame([{"field":col,"message":"Missing column."}])
    for i, x in df[col].items():
        if empty(x):
            continue
        if str(x).strip() not in allowed:
            out.append({"row":i,"field":col,"value":x,"message":f"Allowed: {sorted(allowed)}"})
    return pd.DataFrame(out)

def duplicate_ids(df, col):
    if col not in df.columns:
        return pd.DataFrame()
    d = df[df[col].astype(str).duplicated(keep=False) & df[col].astype(str).str.strip().ne("")]
    return d[[col]].copy()

## 6. Search-coverage audit

In [ ]:
REQUIRED_DOMAINS = [
    "Bahg paradigm","closed-loop recommender feedback","human-AI recursive interaction",
    "real-world platform experiments","normative learning under biased environments",
    "active information seeking","compositional sampling data",
    "reciprocal sampling-learning loops","category-learning systems",
    "learned attention and feature reweighting","measurement reactivity",
    "temporal learning and accumulation","persistence after algorithm removal",
    "sequential/adaptive experimental design","control architectures",
    "static versus dynamic personalisation","correction/reversal/recovery",
    "transfer","computational models of behaviour","model and parameter identifiability",
    "individual differences/susceptibility","statistical/graph learning",
    "experience-dependent representational warping","model-based cognitive neuroscience",
    "fMRI","EEG/ERP","eye tracking","pupillometry","fNIRS","MEG",
    "multimodal neuroscience","neural reliability and power",
    "pilot effects and winner's curse","cross-study adaptive programmes",
    "online versus lab transportability","programme feasibility","novelty audit"
]

def search_coverage_audit(searches):
    out = []
    seen = set(searches["domain"].astype(str).str.strip().str.lower()) if len(searches) else set()
    for d in REQUIRED_DOMAINS:
        if d.lower() not in seen:
            out.append({"domain":d,"severity":"LOAD-BEARING REVIEW","message":"No logged search for required domain."})
    if len(searches):
        for _, r in searches.iterrows():
            if empty(r.get("database","")) or empty(r.get("query","")) or empty(r.get("date_searched","")):
                out.append({"domain":r.get("domain",""),"severity":"TRACEABILITY","message":"Search log missing database, query, or date."})
            if str(r.get("coverage_status","")).upper() == "CLOSED" and empty(r.get("results_screened","")):
                out.append({"domain":r.get("domain",""),"severity":"CLOSURE","message":"Domain marked CLOSED without screening count."})
    return pd.DataFrame(out)

## 7. Source and evidence-claim provenance audit

In [ ]:
def provenance_audit(sources, evidence_claims, propositions):
    out = []
    src_ids = set(sources["evidence_id"].astype(str))
    prop_ids = set(propositions["proposition_id"].astype(str))

    for _, s in sources.iterrows():
        eid = str(s.get("evidence_id",""))
        if empty(s.get("doi","")) and empty(s.get("authoritative_url","")):
            out.append({"target":eid,"type":"SOURCE","message":"No DOI or authoritative URL."})
        if not truthy(s.get("primary_source_verified","")):
            out.append({"target":eid,"type":"SOURCE","message":"Primary/authoritative record not marked verified."})

    for _, e in evidence_claims.iterrows():
        ecid = e.get("evidence_claim_id","")
        if str(e.get("evidence_id","")) not in src_ids:
            out.append({"target":ecid,"type":"EVIDENCE_CLAIM","message":"Evidence ID not found in source registry."})
        if str(e.get("proposition_id","")) not in prop_ids:
            out.append({"target":ecid,"type":"EVIDENCE_CLAIM","message":"Proposition ID not found."})
        if str(e.get("support_relation","")) not in SUPPORT_RELATION:
            out.append({"target":ecid,"type":"EVIDENCE_CLAIM","message":"Invalid support relation."})
        if empty(e.get("what_it_establishes","")) or empty(e.get("what_it_does_not_establish","")):
            out.append({"target":ecid,"type":"EVIDENCE_CLAIM","message":"Must state both what the source establishes and does not establish."})

    # Paper-level citation laundering check: proposition has status but no atomic evidence link
    linked_props = set(evidence_claims["proposition_id"].astype(str)) if len(evidence_claims) else set()
    for _, p in propositions.iterrows():
        pid = str(p.get("proposition_id",""))
        if str(p.get("literature_status","")) in {"SUPPORTED","PARTLY SUPPORTED","CONTRADICTED"} and pid not in linked_props:
            out.append({"target":pid,"type":"PROPOSITION","message":"Literature status assigned without atomic evidence-claim links."})
    return pd.DataFrame(out)

## 8. Typed claim requirements

This is the main logical upgrade over V1.

The notebook does not infer claim type from prose. You specify the scientific claim type explicitly. Each type then has minimum information requirements.

That prevents a vocabulary change from changing the audit outcome.

In [ ]:
def claim_type_audit(propositions, constructs):
    out = []
    c = constructs.set_index("construct_id").to_dict("index") if len(constructs) else {}

    for _, p in propositions.iterrows():
        pid = p.get("proposition_id","")
        typ = str(p.get("claim_type","")).strip()
        if typ not in CLAIM_TYPES:
            out.append({"proposition_id":pid,"severity":"SCHEMA","message":f"Unknown claim_type: {typ}"})
            continue

        # normative claims
        if typ == "normative":
            cid = str(p.get("construct_id",""))
            benchmark = p.get("normative_benchmark_id","")
            cbenchmark = c.get(cid,{}).get("normative_benchmark","") if cid in c else ""
            if empty(benchmark) and empty(cbenchmark):
                out.append({"proposition_id":pid,"severity":"MAJOR","message":"Normative claim lacks an explicit benchmark/reference distribution."})

        # causal claims
        if typ == "total_policy_causal":
            if empty(p.get("estimand","")):
                out.append({"proposition_id":pid,"severity":"MAJOR","message":"Total-policy causal claim lacks explicit estimand."})

        if typ == "component_causal":
            if empty(p.get("estimand","")):
                out.append({"proposition_id":pid,"severity":"ARCHITECTURE-CHANGING","message":"Component-causal claim lacks explicit component estimand."})
            if str(p.get("design_identification_status","")) == "IDENTIFIED" and "random" not in norm_text(p.get("notes","")):
                out.append({"proposition_id":pid,"severity":"MAJOR","message":"Component causal effect marked IDENTIFIED without explicit repeated/component randomisation or equivalent identification logic in notes."})

        if typ == "mediation":
            if empty(p.get("estimand","")):
                out.append({"proposition_id":pid,"severity":"MAJOR","message":"Mediation claim lacks mediation estimand/assumptions."})

        if typ == "moderation":
            if empty(p.get("required_interaction","")):
                out.append({"proposition_id":pid,"severity":"MAJOR","message":"Moderation claim lacks explicit interaction term."})

        if typ == "prediction":
            if empty(p.get("validation_scheme","")):
                out.append({"proposition_id":pid,"severity":"MAJOR","message":"Prediction claim lacks held-out/cross-validated/prospective validation scheme."})

        if typ == "persistence":
            if empty(p.get("time_horizon","")):
                out.append({"proposition_id":pid,"severity":"MAJOR","message":"Persistence claim lacks explicit time horizon."})

        if typ == "recovery":
            if empty(p.get("operationalisation","")):
                out.append({"proposition_id":pid,"severity":"MAJOR","message":"Recovery claim lacks recovery criterion/reference state."})
            if empty(p.get("time_horizon","")):
                out.append({"proposition_id":pid,"severity":"MODERATE","message":"Recovery has no durability horizon; immediate recovery and durable recovery must remain distinct."})

        if typ == "transfer":
            if empty(p.get("transfer_dimensions","")):
                out.append({"proposition_id":pid,"severity":"MAJOR","message":"Transfer claim lacks explicit dimensions (stimulus/context/response/domain/time/etc.)."})

        if typ == "individual_difference":
            cid = str(p.get("construct_id",""))
            rel = c.get(cid,{}).get("reliability_source_ids","") if cid in c else ""
            if empty(rel):
                out.append({"proposition_id":pid,"severity":"MAJOR","message":"Stable individual-difference claim lacks reliability evidence."})

        if typ == "neural_causal":
            if str(p.get("design_identification_status","")) == "IDENTIFIED" and not re.search(r"\b(tms|stimulation|lesion|perturb|causal neural intervention)\b", norm_text(p.get("notes",""))):
                out.append({"proposition_id":pid,"severity":"MAJOR","message":"Neural causal claim marked IDENTIFIED without neural perturbation/other explicit causal neural design."})

        if typ == "novelty":
            if str(p.get("novelty_status","")) == "DIRECTLY_OPEN" and empty(p.get("notes","")):
                out.append({"proposition_id":pid,"severity":"MODERATE","message":"Direct novelty claim requires documented bounded-search basis in notes/search ledger."})

        if typ == "algorithm_mechanics":
            if empty(p.get("operationalisation","")):
                out.append({"proposition_id":pid,"severity":"MAJOR","message":"Algorithm-mechanics claim must specify what updates: parameters, state/history, ranking, candidate set, or policy."})

    return pd.DataFrame(out)

## 9. Construct audit

In [ ]:
def construct_audit(constructs):
    out = []
    for _, c in constructs.iterrows():
        cid = c.get("construct_id","")
        name = norm_text(c.get("construct_name",""))

        if empty(c.get("literature_definition","")):
            out.append({"construct_id":cid,"severity":"MAJOR","message":"No literature definition."})
        if empty(c.get("proposed_operationalisation","")):
            out.append({"construct_id":cid,"severity":"MAJOR","message":"No proposed operationalisation."})

        if any(k in name for k in ["distortion","maladaptive","harm","bias"]):
            if empty(c.get("normative_benchmark","")):
                out.append({"construct_id":cid,"severity":"ARCHITECTURE-CHANGING","message":"Normative construct lacks benchmark/reference distribution."})

        if truthy(c.get("compositional_or_constrained","")):
            notes = norm_text(c.get("notes",""))
            if not any(k in notes for k in ["log ratio","log-ratio","simplex","compositional","ilr","clr","alr"]):
                out.append({"construct_id":cid,"severity":"MAJOR","message":"Constrained/compositional variable has no explicit compositional-analysis treatment."})

        if str(c.get("validity_status","")) == "ADEQUATE" and empty(c.get("validity_source_ids","")):
            out.append({"construct_id":cid,"severity":"MAJOR","message":"Validity marked ADEQUATE without validity evidence IDs."})

    return pd.DataFrame(out)

## 10. Separate evidence status from design capacity

In [ ]:
def status_consistency_audit(propositions):
    out = []
    for _, p in propositions.iterrows():
        pid = p.get("proposition_id","")
        lit = str(p.get("literature_status",""))
        val = str(p.get("construct_validity_status",""))
        ident = str(p.get("design_identification_status",""))

        # A proposition can be literature-supported but not identified by the proposed design.
        # Flag only when prose boundary fails to reflect this.
        strongest = norm_text(p.get("strongest_justified_claim",""))
        forbidden = norm_text(p.get("stronger_claim_not_justified",""))

        if lit == "SUPPORTED" and ident in {"NOT_IDENTIFIED","UNRESOLVED"} and empty(forbidden):
            out.append({"proposition_id":pid,"severity":"MAJOR","message":"Literature supports proposition, but design does not identify it; missing explicit stronger-claim boundary."})

        if val in {"INADEQUATE","UNRESOLVED"} and ident == "IDENTIFIED":
            out.append({"proposition_id":pid,"severity":"MAJOR","message":"Design may identify an operational contrast, but construct validity is inadequate/unresolved. Do not infer the construct merely because the contrast is randomized."})

        if empty(strongest):
            out.append({"proposition_id":pid,"severity":"MODERATE","message":"Missing strongest justified claim."})
        if empty(forbidden):
            out.append({"proposition_id":pid,"severity":"MODERATE","message":"Missing explicit stronger claim that is not justified."})

    return pd.DataFrame(out)

## 11. Semantic-patch audit across versions

In [ ]:
def semantic_patch_audit(old_props, new_props):
    out = []
    old = old_props.copy()
    new = new_props.copy()

    # First: stable IDs
    if "proposition_id" in old.columns and "proposition_id" in new.columns:
        oi = old.set_index("proposition_id")
        ni = new.set_index("proposition_id")
        for pid in oi.index.intersection(ni.index):
            a, b = oi.loc[pid], ni.loc[pid]
            op_sim = fuzz.token_set_ratio(norm_text(a.get("operationalisation","")), norm_text(b.get("operationalisation","")))
            est_sim = fuzz.token_set_ratio(norm_text(a.get("estimand","")), norm_text(b.get("estimand","")))
            claim_sim = fuzz.token_set_ratio(norm_text(a.get("canonical_claim","")), norm_text(b.get("canonical_claim","")))
            type_changed = str(a.get("claim_type","")) != str(b.get("claim_type",""))
            construct_changed = str(a.get("construct_id","")) != str(b.get("construct_id",""))
            if (type_changed or construct_changed or claim_sim < 80) and op_sim >= 85 and est_sim >= 85:
                out.append({
                    "old_id":pid,"new_id":pid,
                    "severity":"MAJOR",
                    "message":"Claim/construct/type changed while operationalisation and estimand stayed essentially unchanged: likely semantic patch.",
                    "op_similarity":op_sim,"estimand_similarity":est_sim,"claim_similarity":claim_sim
                })

    # Second: changed IDs — candidate matching by operationalisation + estimand
    used = set()
    for i, a in old.iterrows():
        for j, b in new.iterrows():
            pair = (i,j)
            if pair in used:
                continue
            op_sim = fuzz.token_set_ratio(norm_text(a.get("operationalisation","")), norm_text(b.get("operationalisation","")))
            est_sim = fuzz.token_set_ratio(norm_text(a.get("estimand","")), norm_text(b.get("estimand","")))
            if op_sim >= 92 and est_sim >= 92 and str(a.get("proposition_id","")) != str(b.get("proposition_id","")):
                if str(a.get("claim_type","")) != str(b.get("claim_type","")) or str(a.get("construct_id","")) != str(b.get("construct_id","")):
                    out.append({
                        "old_id":a.get("proposition_id",""),"new_id":b.get("proposition_id",""),
                        "severity":"MAJOR",
                        "message":"Possible semantic relabelling across changed proposition IDs.",
                        "op_similarity":op_sim,"estimand_similarity":est_sim,"claim_similarity":fuzz.token_set_ratio(norm_text(a.get("canonical_claim","")), norm_text(b.get("canonical_claim","")))
                    })
                used.add(pair)
    return pd.DataFrame(out)

## 12. Semantic contradiction *candidates* — not automatic contradictions

Embedding/similarity systems cannot reliably understand negation, causal direction, or boundary conditions.

V2 therefore uses TF-IDF only to surface **candidate proposition pairs** for human adjudication. It never labels them contradictory automatically.

In [ ]:
def semantic_pair_candidates(propositions, threshold=0.50, max_pairs=250):
    x = propositions.dropna(subset=["canonical_claim"]).copy()
    if len(x) < 2:
        return pd.DataFrame()
    texts = x["canonical_claim"].astype(str).tolist()
    vec = TfidfVectorizer(ngram_range=(1,2), stop_words="english").fit_transform(texts)
    sim = cosine_similarity(vec)
    rows = []
    for i in range(len(x)):
        for j in range(i+1, len(x)):
            if sim[i,j] >= threshold:
                rows.append({
                    "id_1":x.iloc[i].get("proposition_id",""),
                    "claim_1":texts[i],
                    "lit_status_1":x.iloc[i].get("literature_status",""),
                    "id_2":x.iloc[j].get("proposition_id",""),
                    "claim_2":texts[j],
                    "lit_status_2":x.iloc[j].get("literature_status",""),
                    "similarity":round(float(sim[i,j]),3),
                    "action":"MANUAL: same claim, negation, boundary condition, or genuine contradiction?"
                })
    return pd.DataFrame(rows).sort_values("similarity",ascending=False).head(max_pairs) if rows else pd.DataFrame()

## 13. Dependency graph and circularity audit

In [ ]:
def dependency_graph_audit(dependencies, propositions, studies):
    out = []
    G = nx.DiGraph()

    valid_ids = set(propositions["proposition_id"].astype(str)) | set(studies["study_id"].astype(str))
    for _, d in dependencies.iterrows():
        a, b = str(d.get("from_id","")), str(d.get("to_id",""))
        if a not in valid_ids:
            out.append({"target":a,"severity":"MAJOR","message":"Dependency source ID not found."})
        if b not in valid_ids:
            out.append({"target":b,"severity":"MAJOR","message":"Dependency target ID not found."})
        if a and b:
            G.add_edge(a,b,relation=d.get("relation",""))

    try:
        cycles = list(nx.simple_cycles(G))
        for cyc in cycles:
            out.append({"target":" -> ".join(cyc),"severity":"ARCHITECTURE-CHANGING","message":"Circular dependency detected."})
    except Exception:
        pass

    # Load-bearing unsupported proposition feeding a study
    pmap = propositions.set_index("proposition_id").to_dict("index")
    for _, d in dependencies.iterrows():
        a, b = str(d.get("from_id","")), str(d.get("to_id",""))
        if a in pmap and b in set(studies["study_id"].astype(str)):
            p = pmap[a]
            if truthy(d.get("load_bearing","")) and (
                str(p.get("literature_status","")) in {"UNSUPPORTED","CONTRADICTED","UNRESOLVED"} or
                str(p.get("construct_validity_status","")) in {"INADEQUATE","UNRESOLVED"} or
                str(p.get("design_identification_status","")) in {"NOT_IDENTIFIED","UNRESOLVED"}
            ):
                out.append({"target":b,"severity":"ARCHITECTURE-CHANGING","message":f"Study depends on load-bearing proposition {a} that is not secure across evidence/validity/identification."})

    return pd.DataFrame(out), G

## 14. Architecture leakage in Step 1

In [ ]:
DESIGN_PATTERNS = [
    r"\bshould use\b", r"\bshould add\b", r"\bshould replace\b",
    r"\bswitch to\b", r"\bwe recommend\b", r"\btherefore use\b",
    r"\bchoose eeg\b", r"\bchoose fmri\b", r"\bmerge stud",
    r"\bdrop study\b", r"\badd a control\b", r"\bset n\s*=",
    r"\bthe final design is\b", r"\bthe control should be\b"
]

def architecture_leakage(text):
    out = []
    for i, line in enumerate(str(text).splitlines(),1):
        for pat in DESIGN_PATTERNS:
            if re.search(pat,line,flags=re.I):
                out.append({"line":i,"text":line.strip(),"rule":pat,
                            "message":"Potential design prescription inside evidence stage; manually check whether this is evidence description or premature architecture."})
    return pd.DataFrame(out)

## 15. Novelty and absence-language audit

In [ ]:
ABSENCE_PATTERNS = [
    r"\bno studies exist\b", r"\bno research exists\b", r"\bnever been studied\b",
    r"\bfirst study\b", r"\bfirst to\b", r"\bcompletely unexplored\b",
    r"\bunknown in the literature\b"
]

def novelty_language_audit(text):
    out = []
    for i, line in enumerate(str(text).splitlines(),1):
        for pat in ABSENCE_PATTERNS:
            if re.search(pat,line,flags=re.I):
                out.append({"line":i,"text":line.strip(),
                            "message":"Absolute absence/novelty claim. Must be converted to bounded documented-search wording unless an exhaustive review genuinely supports it."})
    return pd.DataFrame(out)

## 16. Cross-study decision / winner's-curse audit

In [ ]:
SELECTION_TERMS = [
    "strongest effect","largest effect","best-performing","most significant",
    "worked best","largest difference","strongest phenotype","highest effect size"
]

def cross_study_selection_audit(text):
    out = []
    for i, line in enumerate(str(text).splitlines(),1):
        low = norm_text(line)
        if any(t in low for t in SELECTION_TERMS):
            out.append({"line":i,"text":line.strip(),"severity":"MAJOR",
                        "message":"Outcome-dependent cross-study selection may induce winner's-curse/researcher-degrees-of-freedom risk. Requires prespecified decision rule and independent confirmation."})
    return pd.DataFrame(out)

## 17. Study-necessity audit structure

In [ ]:
def study_necessity_audit(studies):
    out = []
    allowed = {
        "clearly independently justified",
        "potentially independently justified",
        "weak independent justification",
        "unresolved"
    }
    for _, s in studies.iterrows():
        sid = s.get("study_id","")
        j = norm_text(s.get("independent_workpackage_justification",""))
        if not j:
            out.append({"study_id":sid,"severity":"MAJOR","message":"No explicit independent work-package justification."})
        # Do not decide necessity automatically; enforce explicit reasoning only.
        if empty(s.get("scientific_priority","")):
            out.append({"study_id":sid,"severity":"MODERATE","message":"Scientific priority not stated."})
    return pd.DataFrame(out)

## 18. Run all deterministic audits

In [ ]:
def run_all_audits(searches, sources, evidence_claims, propositions, constructs,
                   studies, dependencies, step1_text=""):
    dep_df, graph = dependency_graph_audit(dependencies, propositions, studies)
    outputs = {
        "search_coverage": search_coverage_audit(searches),
        "provenance": provenance_audit(sources, evidence_claims, propositions),
        "claim_type": claim_type_audit(propositions, constructs),
        "constructs": construct_audit(constructs),
        "status_consistency": status_consistency_audit(propositions),
        "semantic_pair_candidates": semantic_pair_candidates(propositions),
        "dependency_graph": dep_df,
        "architecture_leakage": architecture_leakage(step1_text),
        "novelty_language": novelty_language_audit(step1_text),
        "cross_study_selection": cross_study_selection_audit(step1_text),
        "study_necessity": study_necessity_audit(studies),
        "duplicate_source_ids": duplicate_ids(sources,"evidence_id"),
        "duplicate_proposition_ids": duplicate_ids(propositions,"proposition_id"),
        "duplicate_evidence_claim_ids": duplicate_ids(evidence_claims,"evidence_claim_id"),
    }
    return outputs, graph

def show_audits(outputs):
    for name, df in outputs.items():
        print("\n" + "="*96)
        print(name.upper())
        print("="*96)
        if isinstance(df,pd.DataFrame) and len(df):
            display(df)
        else:
            print("No flags.")

## 19. Closure gate

Step 1 should not be declared closed merely because there are no code flags.

The closure gate has two parts:

### Machine-checkable prerequisites

- all required literature domains have logged searches;
- all load-bearing propositions have atomic evidence links;
- every normative claim has a benchmark;
- every causal claim has an estimand;
- every prediction claim has a validation scheme;
- every persistence claim has a time horizon;
- every transfer claim specifies transfer dimensions;
- unresolved load-bearing dependencies are visible;
- no architecture-changing circular dependency is hidden.

### Scientific/manual prerequisites

A reviewer must still determine:

- whether the literature search was substantively broad enough;
- whether retained sources really support the extracted claims;
- whether constructs are valid;
- whether boundary conditions are correctly interpreted;
- whether apparent contradictions are genuine;
- whether novelty is bounded correctly.

The code can return **MACHINE GATE PASS**. It cannot return **SCIENTIFIC STEP 1 CLOSED** by itself.

In [ ]:
def machine_closure_gate(outputs):
    critical = []
    for name, df in outputs.items():
        if not isinstance(df,pd.DataFrame) or not len(df):
            continue
        if "severity" in df.columns:
            hits = df[df["severity"].astype(str).isin(["MAJOR","ARCHITECTURE-CHANGING","LOAD-BEARING REVIEW"])]
            if len(hits):
                critical.append((name,len(hits)))
        elif name in {"search_coverage","provenance","duplicate_source_ids","duplicate_proposition_ids","duplicate_evidence_claim_ids"}:
            if len(df):
                critical.append((name,len(df)))

    if critical:
        print("MACHINE GATE: NOT READY")
        print("Critical audit groups:", critical)
        return False
    print("MACHINE GATE: PASS")
    print("This is NOT equivalent to scientific Step 1 closure.")
    return True

## 20. Capacity-aware literature discovery helpers

In [ ]:
from Bio import Entrez
Entrez.email = "your_email@example.com"  # CHANGE THIS

def pubmed_search(query, retmax=50):
    handle = Entrez.esearch(db="pubmed", term=query, retmax=retmax, sort="relevance")
    ids = Entrez.read(handle)["IdList"]
    if not ids:
        return pd.DataFrame()
    handle = Entrez.esummary(db="pubmed", id=",".join(ids))
    rows = []
    for r in Entrez.read(handle):
        rows.append({
            "pmid":str(r.get("Id","")),
            "title":str(r.get("Title","")),
            "journal":str(r.get("FullJournalName","")),
            "pubdate":str(r.get("PubDate","")),
            "article_ids":str(r.get("ArticleIds",""))
        })
    return pd.DataFrame(rows)

def openalex_search(query, per_page=50, mailto="your_email@example.com"):
    r = requests.get(
        "https://api.openalex.org/works",
        params={"search":query,"per-page":per_page,"mailto":mailto},
        timeout=30
    )
    r.raise_for_status()
    rows = []
    for w in r.json().get("results",[]):
        rows.append({
            "openalex_id":w.get("id"),
            "title":w.get("display_name"),
            "year":w.get("publication_year"),
            "doi":w.get("doi"),
            "type":w.get("type"),
            "cited_by_count":w.get("cited_by_count"),
            "landing_page":(w.get("primary_location") or {}).get("landing_page_url")
        })
    return pd.DataFrame(rows)

# IMPORTANT:
# These functions generate candidates only.
# They do not establish search completeness and should not replace ACM/IEEE/publisher/manual searches
# for recommender-systems, HCI, neuroscience, or methodological literatures.

## 21. Optional LLM audit — capacity-aware batching

Do **not** send the entire corpus as one giant JSON packet.

For a large Step-1 corpus this creates:
- context-window pressure;
- higher cost;
- source dilution;
- a greater chance that the model cites a paper merely because it is present.

Instead audit one proposition at a time with only its linked evidence claims and source records.

No model name is hardcoded. Use a model actually available to your own API account.

In [ ]:
def build_proposition_packet(proposition_id, propositions, evidence_claims, sources, constructs):
    p = propositions[propositions["proposition_id"].astype(str)==str(proposition_id)]
    if not len(p):
        raise ValueError("Unknown proposition_id")
    ecs = evidence_claims[evidence_claims["proposition_id"].astype(str)==str(proposition_id)]
    eids = set(ecs["evidence_id"].astype(str))
    src = sources[sources["evidence_id"].astype(str).isin(eids)]
    cid = str(p.iloc[0].get("construct_id",""))
    con = constructs[constructs["construct_id"].astype(str)==cid] if cid else constructs.iloc[0:0]
    return {
        "proposition":p.fillna("").to_dict("records"),
        "evidence_claims":ecs.fillna("").to_dict("records"),
        "sources":src.fillna("").to_dict("records"),
        "construct":con.fillna("").to_dict("records")
    }

LLM_AUDIT_INSTRUCTIONS = '''
You are an independent scientific-methods auditor.
Use only the supplied proposition packet.
Do not invent sources.
Do not redesign the study.
Do not solve inferential problems by renaming constructs.
Separate:
(1) literature support,
(2) construct validity,
(3) design identification,
(4) novelty.
For every conclusion cite evidence_claim_id/evidence_id or label it DESIGN LOGIC.
State the strongest justified claim and the stronger claim that is not justified.
Preserve uncertainty and boundary conditions.
'''

def optional_llm_audit_one(proposition_id, propositions, evidence_claims, sources, constructs, model):
    if not model or not str(model).strip():
        raise ValueError("Pass a model name available to your own API account.")
    from openai import OpenAI
    client = OpenAI()
    packet = build_proposition_packet(proposition_id, propositions, evidence_claims, sources, constructs)
    prompt = "Audit this proposition packet:\n" + json.dumps(packet, ensure_ascii=False)
    response = client.responses.create(
        model=model,
        instructions=LLM_AUDIT_INSTRUCTIONS,
        input=prompt
    )
    return response.output_text

## 22. Synthetic unit tests

In [ ]:
def run_synthetic_tests():
    props = pd.DataFrame([
        {
            "proposition_id":"P1","canonical_claim":"Adaptive exposure causes cognitive distortion",
            "claim_type":"normative","construct_id":"C1",
            "operationalisation":"lower balanced-test accuracy","estimand":"A-R accuracy difference",
            "time_horizon":"","transfer_dimensions":"","normative_benchmark_id":"",
            "required_interaction":"","validation_scheme":"",
            "literature_status":"UNRESOLVED","construct_validity_status":"UNRESOLVED",
            "design_identification_status":"IDENTIFIED","novelty_status":"UNRESOLVED",
            "confidence":"","load_bearing":"yes","affected_studies":"S1",
            "strongest_justified_claim":"Adaptive assignment changes balanced-test accuracy",
            "stronger_claim_not_justified":"The learner is intrinsically distorted","notes":""
        },
        {
            "proposition_id":"P2","canonical_claim":"A 24-hour residual effect persists at 24 hours",
            "claim_type":"persistence","construct_id":"C2",
            "operationalisation":"A-R contrast at 24h","estimand":"E[Y24(A)]-E[Y24(R)]",
            "time_horizon":"24 hours","transfer_dimensions":"","normative_benchmark_id":"",
            "required_interaction":"","validation_scheme":"",
            "literature_status":"PARTLY SUPPORTED","construct_validity_status":"ADEQUATE",
            "design_identification_status":"IDENTIFIED","novelty_status":"PARTLY_OPEN",
            "confidence":"","load_bearing":"yes","affected_studies":"S1",
            "strongest_justified_claim":"Residual treatment effect at 24 hours",
            "stronger_claim_not_justified":"Long-term persistence","notes":"randomized policy"
        }
    ])
    cons = pd.DataFrame([
        {"construct_id":"C1","construct_name":"cognitive distortion","literature_definition":"",
         "proposed_operationalisation":"balanced-test accuracy","normative_benchmark":"",
         "benchmark_source_ids":"","validity_source_ids":"","reliability_source_ids":"",
         "compositional_or_constrained":"no","known_failure_modes":"","validity_status":"UNRESOLVED","notes":""},
        {"construct_id":"C2","construct_name":"24-hour residual effect","literature_definition":"difference at specified delay",
         "proposed_operationalisation":"24-hour A-R contrast","normative_benchmark":"",
         "benchmark_source_ids":"","validity_source_ids":"E1","reliability_source_ids":"",
         "compositional_or_constrained":"no","known_failure_modes":"","validity_status":"ADEQUATE","notes":""}
    ])
    a = claim_type_audit(props,cons)
    b = construct_audit(cons)
    assert (a["proposition_id"]=="P1").any(), "P1 normative benchmark problem should be flagged."
    assert not ((a.get("proposition_id",pd.Series(dtype=str))=="P2") & a.get("message",pd.Series(dtype=str)).str.contains("time horizon",case=False,na=False)).any(), "P2 has a 24h horizon and should not get persistence-horizon flag."
    assert (b["construct_id"]=="C1").any(), "C1 construct problems should be flagged."
    print("Synthetic tests passed.")

run_synthetic_tests()

## 23. Export workbook

In [ ]:
def export_audit_workbook(path, datasets, audits):
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        for name, df in datasets.items():
            if isinstance(df,pd.DataFrame):
                df.to_excel(w, sheet_name=re.sub(r"[^A-Za-z0-9_]","_",name)[:31], index=False)
        for name, df in audits.items():
            if isinstance(df,pd.DataFrame):
                sheet = ("AUD_" + re.sub(r"[^A-Za-z0-9_]","_",name))[:31]
                df.to_excel(w, sheet_name=sheet, index=False)
    print(path)

## 24. Recommended operating procedure

1. **Search first.** Log every domain/query/database.
2. **Verify retained sources.** Do not create evidence claims from snippets.
3. **Extract atomic evidence claims.** One paper can support one proposition and counter/qualify another.
4. **Create atomic propositions.** Avoid compound hypotheses.
5. **Define constructs independently of preferred study architecture.**
6. **State estimands separately from construct labels.**
7. **Run deterministic audits.**
8. **Manually adjudicate semantic-pair candidates and contradictions.**
9. **Optionally run proposition-by-proposition LLM audits.**
10. **Create the architecture-impact and recalibration ledgers.**
11. **Run the machine closure gate.**
12. **Only then make the scientific Step-1 closure decision manually.**

The notebook's job is to make hidden assumptions visible. It is not allowed to resolve them by vocabulary.